# Null / Missing Value Analysis — Banorte Cross-Border Transactions

**Dataset**: Banorte XB monthly parquet files (last 4 months)  
**Purpose**: Thorough analysis of null and missing values to inform preprocessing decisions for the SaaS churn prediction pipeline.  
**Date**: 2026-03-05

---

## Table of Contents
1. Setup & Data Loading
2. Null Overview

## 1. Setup & Data Loading

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:,.2f}".format)
sns.set_theme(style="whitegrid", palette="muted")

DATA_DIR = Path("../data/raw/banorte_xb")
N_MONTHS = 4

In [2]:
all_files = sorted(DATA_DIR.rglob("*.parquet"))
files = all_files[-N_MONTHS:]

print(f"Total parquet files available: {len(all_files)}")
print(f"Loading last {N_MONTHS} months:")
for f in files:
    print(f"  {f.relative_to(DATA_DIR)}")

dfs = []
for f in files:
    tmp = pd.read_parquet(f)
    tmp["source_file"] = str(f.relative_to(DATA_DIR))
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

print(f"\nTotal rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")

Total parquet files available: 28
Loading last 4 months:
  2025\12.parquet
  2026\01.parquet
  2026\02.parquet
  2026\03.parquet

Total rows: 5,868,722
Total columns: 20
Memory usage: 7821.2 MB


In [3]:
df.head(3)

,CARD_TYPE,FEE_DATE,CARD_ISSUER,OPERATION_TYPE,AMOUNT_GROSS,AMOUNT_INSTALLMENT,PROVIDER,INSTALLMENTS,PAYMENT_AMOUNT,MERCHANT_NAME,CARD_SCHEME,CONFIRM_DATE,CAMARA_COMPENSACION,CUSTO_TRANSACAO,FEE_EBANX,AMOUNT_IVA_INSTALLMENT,NET_AMOUNT,IVA_EBANX,AFILIACION,source_file
0,CREDIT,2025-12-04,BANCOMER,payment,63.2700000000,0.0000000000,banorte,1,63.2700000000,Temu.com,visa,2025-12-04,BANCOMER,0.017,1.0755900000,0.000000000000,62.0223156000,0.172094400000,09267749,2025\12.parquet
1,DEBIT,2025-12-04,BANCOMER,payment,150.0000000000,0.0000000000,banorte,1,150.0000000000,Canva,visa,2025-12-04,BANCOMER,0.017,2.5500000000,0.000000000000,147.0420000000,0.408000000000,09357577,2025\12.parquet
2,CREDIT,2025-12-04,INVEX,payment,674.6300000000,0.0000000000,banorte,1,674.6300000000,Amazon Retail MX,mastercard,2025-12-04,PROSA,0.017,11.4687100000,0.000000000000,661.3262964000,1.834993600000,08704594,2025\12.parquet


In [4]:
df.dtypes

CARD_TYPE                 object
FEE_DATE                  object
CARD_ISSUER               object
OPERATION_TYPE            object
AMOUNT_GROSS              object
AMOUNT_INSTALLMENT        object
PROVIDER                  object
INSTALLMENTS              object
PAYMENT_AMOUNT            object
MERCHANT_NAME             object
CARD_SCHEME               object
CONFIRM_DATE              object
CAMARA_COMPENSACION       object
CUSTO_TRANSACAO           object
FEE_EBANX                 object
AMOUNT_IVA_INSTALLMENT    object
NET_AMOUNT                object
IVA_EBANX                 object
AFILIACION                object
source_file               object
dtype: object

## 2. Null Overview

Per-column summary of null counts, percentages, and data types.

In [5]:
null_counts = df.isna().sum()
null_pct = (null_counts / len(df)) * 100
non_null_counts = df.isna().sum()

null_summary = pd.DataFrame({
    "dtype": df.dtypes,
    "null_count": null_counts,
    "null_pct": null_pct.round(2),
    "non_null_count": non_null_counts,
}).sort_values("null_pct", ascending=False)

null_summary

,dtype,null_count,null_pct,non_null_count
CARD_ISSUER,object,19809,0.34,19809
CUSTO_TRANSACAO,object,9117,0.16,9117
FEE_DATE,object,0,0.00,0
CARD_TYPE,object,15,0.00,15
OPERATION_TYPE,object,0,0.00,0
AMOUNT_GROSS,object,0,0.00,0
PROVIDER,object,0,0.00,0
AMOUNT_INSTALLMENT,object,0,0.00,0
PAYMENT_AMOUNT,object,0,0.00,0
MERCHANT_NAME,object,0,0.00,0


In [6]:
total_nulls = null_counts.sum()
cols_zero_nulls = (null_counts == 0).sum()
cols_gt50_nulls = (null_pct > 50).sum()
cols_any_nulls = (null_counts > 0).sum()

print(f"Total null values in dataset: {total_nulls:,}")
print(f"Columns with zero nulls:     {cols_zero_nulls}")
print(f"Columns with any nulls:      {cols_any_nulls}")
print(f"Columns with >50% nulls:     {cols_gt50_nulls}")

Total null values in dataset: 28,941
Columns with zero nulls:     17
Columns with any nulls:      3
Columns with >50% nulls:     0


### Recommended Actions

| Category | Strategy | Details |
|----------|----------|---------|
| **Always populated** | No action needed | These columns are reliable features |
| **Partially null (0–50%)** | Investigate & impute | Use domain-appropriate imputation (median, mode, forward-fill) or create a binary `_is_null` indicator feature |
| **Mostly null (>50%)** | Consider dropping | Low information density — drop unless domain knowledge justifies imputation |
| **Correlated nulls** | Investigate root cause | Columns that are null together may share upstream data issues — treat as a group |

**Next steps:**
1. Investigate root cause of high-null columns with the data engineering team
2. For retained columns, test imputation strategies in the modeling pipeline
3. Monitor null trends monthly — a sudden increase signals a data quality regression